<a href="https://colab.research.google.com/github/Soham-1009/GenAI-Practical/blob/main/Practical_7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import math
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Create position matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        # Compute the division term
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        # Even indices (2i) -> sine; Odd indices (2i+1) -> cosine
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add extra dimension for batch compatibility -> (1, max_len, d_model)
        pe = pe.unsqueeze(0)

        # Register pe as buffer so it moves with the model but isn't updated by gradients
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x shape: [batch_size, seq_len, d_model]
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len, :]
        return self.dropout(x)

In [ ]:
class CustomLayerNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        # Learnable parameters
        self.gamma = nn.Parameter(torch.ones(normalized_shape))
        self.beta = nn.Parameter(torch.zeros(normalized_shape))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Compute mean and variance along the last dimension
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Standardize and scale/shift
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return self.gamma * x_norm + self.beta

In [ ]:
class SimpleTransformerEncoderLayer(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_feedforward: int, dropout: float = 0.1):
        super().__init__()
        # Multihead Attention
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)

        # Feed-forward network
        self.linear1 = nn.Linear(d_model, dim_feedforward)
        self.dropout = nn.Dropout(dropout)
        self.linear2 = nn.Linear(dim_feedforward, d_model)

        # Custom Layer Normalizations
        self.norm1 = CustomLayerNorm(d_model)
        self.norm2 = CustomLayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, src: torch.Tensor) -> torch.Tensor:
        # Sub-layer 1: Self-Attention + Residual Connection + LayerNorm
        attn_output, _ = self.self_attn(src, src, src)
        src = src + self.dropout1(attn_output)
        src = self.norm1(src)

        # Sub-layer 2: Feed Forward + Residual Connection + LayerNorm
        ff_output = self.linear2(self.dropout(torch.relu(self.linear1(src))))
        src = src + self.dropout2(ff_output)
        src = self.norm2(src)
        return src

In [ ]:
# Visualizing the Positional Encoding Matrix
d_model, max_len = 128, 60
pe_module = PositionalEncoding(d_model=d_model, max_len=max_len, dropout=0.0)

# Extract buffer for plotting
pe_matrix = pe_module.pe.squeeze(0).cpu().numpy()

plt.figure(figsize=(10, 6))
plt.imshow(pe_matrix, cmap='RdBu', aspect='auto')
plt.xlabel('Embedding Dimension (d_model)')
plt.ylabel('Token Position')
plt.title('Sinusoidal Positional Encoding Matrix')
plt.colorbar(label='Amplitude')
plt.show()

In [ ]:
# Integration Testing and Verification
print("--- Starting Test Suite ---")

batch_size = 4
seq_len = 15
d_model_test = 64

# Instantiate modules
pos_encoder = PositionalEncoding(d_model=d_model_test, max_len=50, dropout=0.1)
custom_ln = CustomLayerNorm(normalized_shape=d_model_test)
encoder_layer = SimpleTransformerEncoderLayer(d_model=d_model_test, nhead=4, dim_feedforward=256)

# Create raw dummy input
x_dummy = torch.randn(batch_size, seq_len, d_model_test)
print(f"Raw Inputs: Shape = {x_dummy.shape}")

# Test Positional Encoding
x_pe = pos_encoder(x_dummy)
print(f"After Positional Encoding: Shape = {x_pe.shape} (Matches Raw Inputs)")

# Test Layer Norm Statistics
x_norm = custom_ln(x_pe)
mean_norm = x_norm.mean(dim=-1).detach().numpy()
var_norm = x_norm.var(dim=-1, unbiased=False).detach().numpy()
print(f"LayerNorm Output Mean (Ideal: ~0): {mean_norm.mean():.4f}")
print(f"LayerNorm Output Var  (Ideal: ~1): {var_norm.mean():.4f}")

# Test full Encoder layer forward pass
out = encoder_layer(x_pe)
print(f"Encoder output: Shape = {out.shape} (Matches Expected Output Shape)")
print("--- Tests Passed Successfully! ---")